In [ ]:
'''
Projeto 8/15 — Análise com múltiplas tabelas (JOIN)
Objetivo


Dataset
↓
Pandas
↓
Criar múltiplas tabelas
↓
SQLite
↓
Queries com JOIN
↓
Pandas
↓
Visualizações
Cenário de negócio

Vamos dividir o dataset em 3 tabelas.

Etapa 1 — Criar as tabelas

A partir do dataset Superstore, você deve criar:

Tabela 1 — Orders

Campos possíveis:

order_id
order_date
ship_date
customer_id
product_id
quantity
sales
profit
region
Tabela 2 — Customers

Campos possíveis:

customer_id
customer_name
segment
region
state
city
Tabela 3 — Products

Campos possíveis:

product_id
product_name
category
sub_category

⚠️ Importante:

Você terá que criar alguns IDs, porque o dataset não vem perfeitamente normalizado.

Isso já é uma boa prática de modelagem simples.

Etapa 2 — Criar banco SQLite

Criar banco:

ecommerce.db

E salvar as tabelas:

orders
customers
products
Etapa 3 — Validar as tabelas

Etapa 4 — Fazer JOINs

Agora começa a parte principal do projeto.

Você deve praticar:

INNER JOIN
LEFT JOIN

Exemplo conceitual:

orders
JOIN customers
JOIN products
Etapa 5 — Perguntas analíticas

Você deve responder perguntas como:

1️⃣ Qual categoria gera mais receita?

Isso exige:

orders + products
2️⃣ Qual segmento de cliente gera mais lucro?

Isso exige:

orders + customers
3️⃣ Quais são os produtos mais vendidos por quantidade?

Isso exige:

orders + products
4️⃣ Qual região tem maior faturamento?

Isso exige:

orders + customers
5️⃣ Quais categorias vendem mais em cada região?

Isso exige:

orders + products + customers

Agora você já está fazendo joins múltiplos.

Etapa 6 — Trazer resultados para Pandas

Depois das queries:

SQL
↓
DataFrame

E criar visualizações.

Exemplos:

categoria x receita
segmento x lucro
região x vendas
Etapa 7 — Visualizações

Sugestões:

análise	gráfico
categoria receita	barplot
segmento lucro	barplot
região vendas	barplot
tempo vendas	lineplot
'''

In [ ]:
#Ingestion
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vivek468/superstore-dataset-final")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Hyego Jarllys\.cache\kagglehub\datasets\vivek468\superstore-dataset-final\versions\1


In [3]:
import os

arquivo= os.listdir(path)
print (f'arquivo{arquivo}')
caminho=r'C:\Users\Hyego Jarllys\.cache\kagglehub\datasets\vivek468\superstore-dataset-final\versions\1\Sample - Superstore.csv'

arquivo['Sample - Superstore.csv']


In [ ]:
#Tratamento dos dados(limpeza)
import pandas as pd
import matplotlib.pyplot as plt

df=pd.read_csv(caminho, encoding='windows-1252')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [9]:
df.drop_duplicates()
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [13]:

df['Ship Date']= pd.to_datetime(df["Ship Date"])
df['Order Date']= pd.to_datetime(df["Order Date"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category   9994 non-null   s

In [ ]:
#Normalização
df_orders= df[['Order ID','Order Date','Ship Date', 'Ship Mode', 'Customer ID', 'Product ID', 'Sales','Quantity','Discount','Profit']]
df_customer=df[['Customer ID','Customer Name','Country','City','State','Postal Code', 'Region']]
df_product=df[['Product ID','Product Name','Segment','Category','Sub-Category']]